# Exp 023 — Rerank + Qwen 3B + longer responses (Blind-A ship)

**Option B — sophisticated zero-training stack.** Three orthogonal changes vs exp 021 baseline (stock prompt, Qwen 1.5B, wRRF, composite 0.33 LLM 3.15):

1. **Qwen 2.5-1.5B → Qwen 2.5-3B** (response LM) — more coherent long responses, better citation, ~2× VRAM on A100 but still well under 40 GB.
2. **BGE-reranker-v2-m3 cross-encoder** over wRRF top-40 → top-20 submission. Single-model reranker, bf16 on CUDA, ~568M params. Addresses the 52% retrieval gap.
3. **`max_new_tokens` 64 → 192** — fixes the hard cap (prior observations: median response 40 words, prior-branch v10 at 192 tokens scored 81 words / LLM 3.25).

**Stock response prompt retained** — exp 022 falsified the persona prompt on Qwen 1.5B. Keep what's proven.

Expected: composite 0.33 → **~0.39–0.41**; nDCG@20 0.19 → 0.22–0.25; LLM 3.15 → 3.3–3.6.

Wall time: ~5–8 min on A100 (Qwen 3B ~2× slower than 1.5B; BGE-reranker adds ~30 s; 80 rows).

Output → `/content/prediction.zip` (CodaBench-compliant) + Drive copy tagged with TID.

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) FORCE-FRESH clone fresh-model.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial

print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%nauthor:  %an%ndate:    %ai%nsubject: %s'
print()
!echo -n 'branch:  ' && git rev-parse --abbrev-ref HEAD

In [ ]:
# 3) Install pinned deps.
!pip install -q -r requirements.txt
!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 4) Experiment parameters (baked for exp 023).
TID = '023-rerank-qwen3b-blindsetA'
BATCH_SIZE = 16  # Qwen 3B + reranker = more VRAM per batch; conservative.
ATTN = 'sdpa'
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Run Blind-A inference with reranker + Qwen 3B + max_new_tokens=192.
# First run will download Qwen 2.5-3B (~6 GB) and BGE-reranker-v2-m3 (~1.1 GB)
# plus build a tid-text cache for the reranker (~30 s). Subsequent runs in the
# same Colab session reuse them.
!cd music-crs-baselines && PYTORCH_ALLOC_CONF=expandable_segments:True \
    python run_inference_blindset.py \
    --tid {TID} \
    --eval_dataset blindset_A \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 6) Validate + package CodaBench-compliant prediction.zip.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/blindset_A/{TID}.json'
assert os.path.isfile(SRC), f'inference output missing at {SRC}'
with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)} (expected 80)')
assert len(rows) == 80
sample = rows[0]
required = {'session_id','user_id','turn_number','predicted_track_ids','predicted_response'}
assert not (required - set(sample.keys()))
assert len(sample['predicted_track_ids']) == 20
assert sample['predicted_response'].strip()

# Length/quality sanity — we expect markedly longer responses now (max_new_tokens=192).
word_lens = sorted(len(r['predicted_response'].split()) for r in rows)
print(f'response words: p25={word_lens[20]} median={word_lens[40]} p75={word_lens[60]} max={word_lens[-1]}')
print(f'(021 baseline at max_new=64: median 40 words. Expect median 80+ now.)')
print(f'sample response[0]: {sample["predicted_response"][:300]!r}')

stage = '/content/_stage_prediction'
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, 'prediction.json'))
!cd {stage} && rm -f /content/prediction.zip && zip -q /content/prediction.zip prediction.json
!unzip -l /content/prediction.zip
print('\nprediction.zip ready — CodaBench-compliant.')

In [ ]:
# 7a) Browser download of prediction.zip.
from google.colab import files
files.download('/content/prediction.zip')

In [ ]:
# 7b) Drive backup — both the CodaBench zip and the raw tagged JSON.
from google.colab import drive
import os, shutil
drive.mount('/content/drive')
dst = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst, exist_ok=True)
shutil.copy('/content/prediction.zip', f'{dst}/{TID}__prediction.zip')
shutil.copy(f'music-crs-baselines/exp/inference/blindset_A/{TID}.json', dst)
!ls -lh {dst}

## Ship `prediction.zip` to CodaBench.

After scoring, ping Claude with `(composite, nDCG@20, LexDiv, LLM)`. I'll:
- Append the `[blindA]` row to `documents/submissions_log.md`
- Decompose Δ vs 021 (stock baseline) across the three axes — we can partially attribute each component's lift since they touch orthogonal metrics:
  - **Rerank** → nDCG@20 (retrieval only, doesn't touch response)
  - **Qwen 3B** → LLM judge + LexDiv (generation only, doesn't touch retrieval)
  - **Longer responses** → LLM judge + LexDiv (via more metadata citations)
- Stage exp 024 based on which axis over/under-delivered.